In [ ]:
# ==============================================================================
# 🚀 DO NOT MODIFY: Standardized Notebook Setup
# ==============================================================================
# This cell is designed to work in both Google Colab and local environments.
# It ensures that the environment is correctly configured by cloning (or
# locating) the project repository and installing the necessary dependencies.
#
# ------------------------------------------------------------------------------
#
#  ⚠️  IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY (NOT ON COLAB):
#
#  This cell will automatically find the repository root and configure your
#  environment. Just make sure you have run: pip install -e .[dev]
#
# ------------------------------------------------------------------------------

import os
import subprocess
import sys
from pathlib import Path

# --- Configuration ---
REPO_URL = "https://github.com/BradSegal/ADH-LLM-Tutorials-2025.git"
REPO_DIR = Path("ADH-LLM-Tutorials-2025")  # The name of the directory once cloned
# --- End of Configuration ---


def find_repo_root(start_path: Path) -> Path | None:
    """
    Find the repository root by looking for pyproject.toml.

    Searches upward from start_path until it finds pyproject.toml or hits root.

    Args:
        start_path: Directory to start searching from.

    Returns:
        Path to repository root, or None if not found.
    """
    current = start_path.resolve()
    while current != current.parent:  # Stop at filesystem root
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    return None


def detect_active_branch(repo_dir: Path) -> str:
    """
    Determine the active git branch for pulling updates.

    Tries multiple methods to detect the current branch name.

    Args:
        repo_dir: Path to the git repository.

    Returns:
        Branch name (defaults to 'master' if detection fails).
    """
    commands = [
        "git symbolic-ref --short HEAD",
        "git rev-parse --abbrev-ref HEAD",
    ]
    for cmd in commands:
        result = subprocess.run(
            cmd, shell=True, cwd=repo_dir, capture_output=True, text=True
        )
        if result.returncode == 0:
            branch = result.stdout.strip()
            if branch and not branch.startswith("origin/"):
                return branch
    return "master"


def run_cmd(cmd: str, *, cwd: Path | None = None) -> None:
    """
    Run a shell command and raise an error if it fails.

    Args:
        cmd: The command to run.
        cwd: Optional working directory for the command.

    Raises:
        RuntimeError: If the command returns a non-zero exit code.
    """
    result = subprocess.run(cmd, shell=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")


# --- Detect environment ---
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --- Main setup logic ---
if IN_COLAB:
    print("☁️  Running in Google Colab. Setting up the environment...\n")

    # Determine repository path
    start_dir = Path.cwd()
    if start_dir.name == REPO_DIR.name:
        repo_path = start_dir
    else:
        repo_path = start_dir / REPO_DIR

    # Clone or update repository
    if not repo_path.exists():
        print(f"📥 Cloning repository from {REPO_URL}...")
        run_cmd(f"git clone --quiet {REPO_URL} {repo_path}")
        print(f"✅ Repository cloned to {repo_path}\n")
    else:
        print(f"📂 Repository already exists at {repo_path}")
        active_branch = detect_active_branch(repo_path)
        print(f"🔄 Pulling latest changes from branch '{active_branch}'...")
        run_cmd(f"git pull origin {active_branch} --quiet", cwd=repo_path)
        print(f"✅ Repository updated\n")

    # Verify repository structure
    if not (repo_path / "pyproject.toml").exists():
        raise FileNotFoundError(
            f"Repository structure invalid: pyproject.toml not found in {repo_path}. "
            "The repository may be corrupted."
        )

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    # Install dependencies
    print("📦 Installing dependencies (this may take a minute)...")
    run_cmd("pip install -q -e .[dev]", cwd=repo_path)

    print("\n" + "=" * 70)
    print("✅ Environment setup complete! You can now proceed with the notebook.")
    print("=" * 70)

else:
    print("💻 Running in local environment. Configuring...\n")

    # Find the repository root
    repo_path = find_repo_root(Path.cwd())

    if repo_path is None:
        raise FileNotFoundError(
            "Could not find repository root (no pyproject.toml found). "
            "Please ensure you are running this notebook from within the "
            "ADH-LLM-Tutorials-2025 repository directory."
        )

    print(f"✅ Found repository root: {repo_path}")

    # Change working directory and update Python path
    print(f"📁 Changing working directory to {repo_path}")
    os.chdir(repo_path)

    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))

    print("\n" + "=" * 70)
    print("✅ Local environment configured successfully!")
    print("=" * 70)
    print("\n⚠️  Please ensure you have run: pip install -e .[dev]")
    print("   (Required for local development)")


# 07 - Understanding Embeddings: The Foundation of LLMs

## What Are Embeddings?

In this notebook, you'll learn about **embeddings**, which are the numerical representations that enable machines to understand and process text. At their core, embeddings convert words, sentences, or entire documents into vectors of numbers that capture their semantic meaning.

**Key Insight:** Similar concepts have similar embeddings. For example, "heart attack" and "myocardial infarction" should have very similar numerical representations because they refer to the same medical condition.

### Learning Objectives

By the end of this notebook, you will:
1. Generate embeddings for clinical terms using a pre-trained model
2. Visualize high-dimensional embeddings in 2D space using UMAP
3. Observe that semantically related terms cluster together

Let's begin by seeing this concept in action!

In [ ]:
# Import required libraries
import plotly.express as px
import umap

from core.llm import EmbeddingRequest, generate_embeddings

## Step 1: Define a Vocabulary of Clinical Concepts

We'll start with a comprehensive set of clinical terms representing different medical domains. Notice how the vocabulary includes:
- **Medical jargon vs. lay terms** (e.g., "Myocardial infarction" vs. "Heart attack")
- **Generic vs. brand names** for medications (e.g., "Acetaminophen" vs. "Tylenol")
- **Abbreviations vs. full terms** (e.g., "MI" vs. "Myocardial infarction")
- **Related symptoms and conditions** across multiple body systems

We've categorized each term by medical domain to help us visualize semantic clustering later.

In [ ]:
# Define clinical vocabulary as a dictionary: {term: category}
clinical_vocabulary = {
    # Cardiovascular - Conditions
    "Myocardial infarction": "Cardiovascular",
    "Heart attack": "Cardiovascular",
    "MI": "Cardiovascular",
    "Acute coronary syndrome": "Cardiovascular",
    "Angina pectoris": "Cardiovascular",
    "Chest pain": "Cardiovascular",
    "Hypertension": "Cardiovascular",
    "High blood pressure": "Cardiovascular",
    "Atrial fibrillation": "Cardiovascular",
    "Irregular heartbeat": "Cardiovascular",
    "Heart failure": "Cardiovascular",
    "Congestive heart failure": "Cardiovascular",
    "Stroke": "Cardiovascular",
    "Cerebrovascular accident": "Cardiovascular",
    # Endocrine/Metabolic
    "Type 2 diabetes": "Endocrine",
    "Diabetes mellitus": "Endocrine",
    "High blood sugar": "Endocrine",
    "Hyperglycemia": "Endocrine",
    "Insulin resistance": "Endocrine",
    "Hypoglycemia": "Endocrine",
    "Low blood sugar": "Endocrine",
    "Thyroid disorder": "Endocrine",
    "Hypothyroidism": "Endocrine",
    "Hyperthyroidism": "Endocrine",
    # Renal/Urinary
    "Renal failure": "Renal",
    "Kidney disease": "Renal",
    "Chronic kidney disease": "Renal",
    "CKD": "Renal",
    "Acute kidney injury": "Renal",
    "Urinary tract infection": "Renal",
    "UTI": "Renal",
    # Respiratory
    "Pneumonia": "Respiratory",
    "Lung infection": "Respiratory",
    "Chronic obstructive pulmonary disease": "Respiratory",
    "COPD": "Respiratory",
    "Asthma": "Respiratory",
    "Shortness of breath": "Respiratory",
    "Dyspnea": "Respiratory",
    "Pulmonary embolism": "Respiratory",
    # Neurological
    "Seizure": "Neurological",
    "Epilepsy": "Neurological",
    "Migraine": "Neurological",
    "Severe headache": "Neurological",
    "Dementia": "Neurological",
    "Alzheimer's disease": "Neurological",
    "Parkinson's disease": "Neurological",
    # Mental Health
    "Depression": "Mental Health",
    "Major depressive disorder": "Mental Health",
    "Anxiety": "Mental Health",
    "Panic attack": "Mental Health",
    "Post-traumatic stress disorder": "Mental Health",
    "PTSD": "Mental Health",
    # Gastrointestinal
    "Gastroesophageal reflux disease": "Gastrointestinal",
    "GERD": "Gastrointestinal",
    "Acid reflux": "Gastrointestinal",
    "Heartburn": "Gastrointestinal",
    "Inflammatory bowel disease": "Gastrointestinal",
    "IBD": "Gastrointestinal",
    "Crohn's disease": "Gastrointestinal",
    # Medications - Cardiovascular
    "Aspirin": "Medication",
    "Acetylsalicylic acid": "Medication",
    "Atorvastatin": "Medication",
    "Lipitor": "Medication",
    "Metoprolol": "Medication",
    "Beta blocker": "Medication",
    # Medications - Other
    "Acetaminophen": "Medication",
    "Tylenol": "Medication",
    "Ibuprofen": "Medication",
    "Advil": "Medication",
    "Metformin": "Medication",
    "Insulin": "Medication",
    # Procedures
    "Angioplasty": "Procedure",
    "Coronary stenting": "Procedure",
    "Cardiac catheterization": "Procedure",
    "Bypass surgery": "Procedure",
    "CABG": "Procedure",
    "Appendectomy": "Procedure",
    "CT scan": "Procedure",
    "Computed tomography": "Procedure",
    "MRI": "Procedure",
    "Magnetic resonance imaging": "Procedure",
    # Lab Tests/Values
    "Hemoglobin A1c": "Lab Test",
    "HbA1c": "Lab Test",
    "Blood glucose": "Lab Test",
    "Serum creatinine": "Lab Test",
    "Cholesterol": "Lab Test",
    "LDL": "Lab Test",
    "Troponin": "Lab Test",
    # General Symptoms
    "Fever": "General Symptom",
    "Elevated temperature": "General Symptom",
    "Nausea": "General Symptom",
    "Vomiting": "General Symptom",
    "Fatigue": "General Symptom",
    "Exhaustion": "General Symptom",
    "Dizziness": "General Symptom",
    "Vertigo": "General Symptom",
}

# Extract terms and categories for processing
clinical_terms = list(clinical_vocabulary.keys())
term_categories = list(clinical_vocabulary.values())

print(f"Total terms in vocabulary: {len(clinical_terms)}")
print(f"Unique categories: {len(set(term_categories))}")
print("\nCategory distribution:")
from collections import Counter

category_counts = Counter(term_categories)
for category, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {category}: {count} terms")

print("\nSample terms from each category:")
seen_categories = set()
for term, category in clinical_vocabulary.items():
    if category not in seen_categories:
        print(f"  [{category}] {term}")
        seen_categories.add(category)
    if len(seen_categories) >= 8:
        break

## Step 2: Generate Embeddings

Now we'll convert each clinical term into a numerical vector using a pre-trained embedding model. The model we're using (`all-MiniLM-L6-v2`) is a lightweight model from the `sentence-transformers` library that produces 384-dimensional vectors.

**What's happening behind the scenes:**
- The model has been trained on millions of sentences to learn semantic relationships
- Each term is encoded into a 384-dimensional vector
- Similar terms will have similar vectors (measured by cosine similarity)

In [ ]:
# Generate embeddings for all clinical terms
embedding_request = EmbeddingRequest(texts=clinical_terms)
embedding_response = generate_embeddings(embedding_request)
embeddings = embedding_response.vectors

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"  - {embeddings.shape[0]} terms")
print(f"  - {embeddings.shape[1]} dimensions per embedding")
print(f"Model used: {embedding_response.model_name}")
print(f"\nFirst embedding (truncated): {embeddings[0][:10]}...")

## Step 3: Reduce Dimensionality and Visualize

While 384 dimensions capture rich semantic information, humans can't visualize that many dimensions! We'll use **UMAP (Uniform Manifold Approximation and Projection)** to project our embeddings down to 2D while preserving their relative relationships.

**Key Question to Consider:**
After running this visualization, do you see clustering of related terms? Are "Myocardial infarction" and "Heart attack" close together?

In [ ]:
# Reduce embeddings to 2D using UMAP
# With more terms, we use more neighbors for better global structure
reducer = umap.UMAP(
    n_components=2, random_state=42, n_neighbors=15, min_dist=0.1, metric="cosine"
)
embeddings_2d = reducer.fit_transform(embeddings)

print(f"Reduced embeddings shape: {embeddings_2d.shape}")

# Create an interactive scatter plot
fig = px.scatter(
    x=embeddings_2d[:, 0],
    y=embeddings_2d[:, 1],
    color=term_categories,
    text=clinical_terms,
    labels={"x": "UMAP Dimension 1", "y": "UMAP Dimension 2", "color": "Category"},
    title="2D Visualization of Clinical Term Embeddings",
    width=1100,
    height=800,
)

fig.update_traces(
    textposition="top center",
    marker={"size": 10, "line": {"width": 1.5, "color": "white"}},
    textfont={"size": 9},
)

fig.update_layout(
    font={"size": 11},
    title_font_size=16,
    showlegend=True,
    legend={
        "title": "Medical Domain",
        "orientation": "v",
        "yanchor": "top",
        "y": 1,
        "xanchor": "left",
        "x": 1.02,
    },
)

fig.show()

## Analysis: What Do You Observe?

### Key Observations to Make

1. **Semantic Clustering of Synonyms:**
   - "Myocardial infarction," "Heart attack," and "MI" should cluster very tightly (same condition, different terminology)
   - "Hypertension" and "High blood pressure" should be nearly identical in embedding space
   - "Acetaminophen" and "Tylenol" (generic vs. brand name) should be very close
   - "COPD" and "Chronic obstructive pulmonary disease" should overlap

2. **Domain-Level Separation:**
   - **Cardiovascular cluster:** Heart attack, stroke, hypertension, atrial fibrillation
   - **Endocrine cluster:** Diabetes, insulin, blood sugar terms
   - **Renal cluster:** Kidney disease, CKD, renal failure
   - **Respiratory cluster:** Pneumonia, COPD, asthma, dyspnea
   - **Neurological cluster:** Seizure, migraine, dementia, Alzheimer's
   - **Mental health cluster:** Depression, anxiety, PTSD
   - **Procedure cluster:** Angioplasty, CT scan, MRI
   - **Medication cluster:** May form sub-clusters (cardiovascular drugs vs. pain relievers)

3. **Cross-Domain Relationships:**
   - Some terms may bridge categories (e.g., "Insulin" is both a medication and endocrine-related)
   - General symptoms (fever, fatigue) may appear between clusters or in their own region
   - Lab tests might cluster near the conditions they diagnose (e.g., "HbA1c" near diabetes terms)

4. **Medical vs. Lay Terminology:**
   - Medical jargon and lay terms for the same condition cluster together, showing the model understands they're semantically equivalent
   - This is critical for clinical applications where patients use different language than providers

### The Power of Pre-Training

Remember: we didn't teach this model anything about medicine! The model learned these semantic relationships from being trained on large amounts of general text, including medical literature, patient forums, and health websites.

### Why This Matters

Embeddings capture meaning in a way that makes **semantic similarity** measurable through geometric proximity. This property is fundamental to:
- **Semantic search:** Find relevant information even when different words are used
- **Clinical decision support:** Match patient symptoms to potential diagnoses
- **Medical coding:** Automatically suggest ICD or CPT codes
- **Literature review:** Find related research papers
- **Patient matching:** Identify similar patient cases

In the next notebook, we'll put this concept to practical use by building a semantic search engine over clinical notes.

## Extension: Explore Different Terms (Optional)

Try modifying the `clinical_vocabulary` dictionary to explore different medical concepts! The dictionary format makes it easy to add new terms.

### Suggested Experiments

1. **Add more synonym pairs:**
   ```python
   "Fracture": "Musculoskeletal",
   "Broken bone": "Musculoskeletal",
   ```

2. **Explore oncology terms:**
   ```python
   "Cancer": "Oncology",
   "Malignant neoplasm": "Oncology",
   "Chemotherapy": "Oncology",
   ```

3. **Add infectious diseases:**
   ```python
   "COVID-19": "Infectious Disease",
   "SARS-CoV-2 infection": "Infectious Disease",
   "Influenza": "Infectious Disease",
   "The flu": "Infectious Disease",
   ```

4. **Include surgical procedures:**
   ```python
   "Cholecystectomy": "Surgery",
   "Gallbladder removal": "Surgery",
   ```

5. **Test multilingual equivalents** (if your embedding model supports it):
   ```python
   "Diabetes": "Endocrine",
   "Diabète": "Endocrine",  # French
   ```

After adding your terms, re-run the cells to see how the visualization changes! Pay attention to:
- Do your new synonyms cluster together?
- Do medical abbreviations cluster with their full forms?
- How do brand names relate to generic drug names?